# Changer de base de couleurs

| Module | Durée | Prérequis |
|---|---|---|
| M5 – Espaces vectoriels et bases | 75 min | M4 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Le club photo du lycée héberge ses images sur un petit serveur presque plein. Le trésorier propose de **diviser par deux** la place occupée par chaque photo. Deux membres du club proposent chacun une méthode, pour le même gain de place :

- **Inès** garde les trois canaux rouge, vert, bleu, mais ne conserve qu'une colonne de pixels sur deux dans chacun d'eux ;
- **Yanis** convertit d'abord l'image dans un autre système de couleurs, appelé **YCbCr**, puis garde la luminosité $Y$ en pleine résolution et réduit fortement les deux autres canaux.

C'est la méthode de Yanis qu'utilise le format JPEG, présent dans tous les téléphones.

**La question du club : à place égale, quelle méthode abîme le moins les photos ? Et pourquoi le simple fait de changer de système de couleurs peut-il faire gagner de la qualité ?**

**Activité** : Comment mesurer qu'une image est « abîmée » ? Un œil humain suffit-il, ou faut-il un nombre ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_meilleure_methode = "Inès"     # "Inès" ou "Yanis" (modifiez)
pari_ecart_qualite = "faible"       # "faible", "net" ou "énorme" (modifiez)

## ③ Abstraire : coordonnées et changement de base

### Un même vecteur, plusieurs jeux de coordonnées

Une couleur est un vecteur de $\mathbb{R}^3$. Ses composantes $(R\,;\,G\,;\,B)$ sont ses **coordonnées dans la base canonique** : rouge pur, vert pur, bleu pur. Mais on peut décrire le même vecteur avec d'autres vecteurs de référence, à condition qu'ils forment une **base** : ils doivent être linéairement indépendants (aucun n'est combinaison des autres) et assez nombreux pour atteindre tout l'espace.

![Schéma : le même vecteur v a pour coordonnées (3 ; 3) dans la base canonique et (0,5 ; 2) dans la base f1, f2](../../assets/schemas/schema-changement-base.svg)

Si les vecteurs de la nouvelle base sont les colonnes d'une matrice $P$, les nouvelles coordonnées $\mathbf{c}$ d'un vecteur $\mathbf{v}$ vérifient

$$
P\,\mathbf{c} = \mathbf{v} \qquad\text{donc}\qquad \mathbf{c} = P^{-1}\mathbf{v}.
$$

Changer de base, c'est résoudre un système : ce qu'on sait faire depuis le module M2. La matrice $P$ est inversible précisément **parce que** ses colonnes forment une base (module M4).

**Activité** : Avec $\mathbf{f}_1 = (2\,;\,0)$ et $\mathbf{f}_2 = (1\,;\,1{,}5)$, retrouvez les coordonnées $(0{,}5\,;\,2)$ du vecteur $\mathbf{v} = (3\,;\,3)$ en résolvant $c_1\mathbf{f}_1 + c_2\mathbf{f}_2 = \mathbf{v}$.

Indiquez vos réponses et justifiez-les.

### La base de couleurs YCbCr

La norme JPEG définit trois nouvelles coordonnées :

$$
\begin{pmatrix} Y \\ C_b \\ C_r \end{pmatrix}
=
\underbrace{\begin{pmatrix}
 0{,}299 & 0{,}587 & 0{,}114 \\
-0{,}1687 & -0{,}3313 & 0{,}5 \\
 0{,}5 & -0{,}4187 & -0{,}0813
\end{pmatrix}}_{M}
\begin{pmatrix} R \\ G \\ B \end{pmatrix}
+
\begin{pmatrix} 0 \\ 128 \\ 128 \end{pmatrix}
$$

- $Y$ est la **luminance** : une moyenne pondérée de $R$, $G$, $B$. Le vert pèse le plus, car l'œil y est le plus sensible.
- $C_b$ et $C_r$ mesurent l'écart au gris vers le bleu et vers le rouge : ils portent la **teinte**.

![Schéma : le cube des couleurs RVB ; la diagonale du noir au blanc porte la luminance, les directions perpendiculaires portent la teinte](../../assets/schemas/schema-cube-rvb.svg)

**Le concept s'applique-t-il ?** La matrice $M$ donne les nouvelles coordonnées à partir des anciennes : c'est donc $P^{-1}$, et la nouvelle base est formée des **colonnes de** $M^{-1}$. Le décalage de 128 sert seulement à ce que $C_b$ et $C_r$ restent entre 0 et 255 ; il ne change rien au raisonnement.

**Activité** : Que valent $C_b$ et $C_r$ pour un gris, c'est-à-dire quand $R = G = B$ ? Vérifiez sur la matrice (la somme de chacune des deux dernières lignes).

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### La photo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def charger_photo():
    try:
        from matplotlib import cbook
        with cbook.get_sample_data("grace_hopper.jpg") as f:
            return plt.imread(f).astype(float)
    except Exception:
        # Image de secours, générée par le calcul, si la photo d'exemple est absente
        y, x = np.mgrid[0:600, 0:512] / 512
        r = 128 + 100 * np.sin(6 * x) * np.cos(4 * y)
        g = 128 + 90 * np.cos(9 * x * y)
        b = 128 + 100 * np.sin(5 * (x + y)) + 20 * ((x * 40).astype(int) % 2)
        return np.clip(np.dstack([r, g, b]), 0, 255)

photo = charger_photo()
print("Taille :", photo.shape, " (hauteur, largeur, canaux)")
plt.figure(figsize=(4, 4.6)); plt.imshow(photo.astype(np.uint8)); plt.axis("off")
plt.title("Grace Hopper, pionnière de l'informatique"); plt.show()

### Le changement de base, appliqué à tous les pixels d'un coup

Chaque pixel est un vecteur. On range les pixels en lignes d'un tableau `(nombre de pixels, 3)` : multiplier par $M$ chaque pixel revient alors à multiplier ce tableau par $M^{\mathsf{T}}$ à droite.

In [ ]:
M = np.array([[ 0.299,     0.587,     0.114   ],
              [-0.168736, -0.331264,  0.5     ],
              [ 0.5,      -0.418688, -0.081312]])
decalage = np.array([0, 128, 128])

def rvb_vers_ycbcr(image):
    return image @ M.T + decalage

def ycbcr_vers_rvb(image):
    return (image - decalage) @ np.linalg.inv(M).T

ycc = rvb_vers_ycbcr(photo)
retour = ycbcr_vers_rvb(ycc)
print("Aller-retour exact ?", np.allclose(retour, photo))
print("\nUn gris (100, 100, 100) ->", rvb_vers_ycbcr(np.array([100., 100, 100])).round(3))
print("Un rouge (200, 30, 30)   ->", rvb_vers_ycbcr(np.array([200., 30, 30])).round(1))

### Les trois canaux de chaque base

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 8))
for k, nom in enumerate(["R", "G", "B"]):
    axes[0, k].imshow(photo[:, :, k], cmap="gray", vmin=0, vmax=255)
    axes[0, k].set_title(f"{nom} (écart type {photo[:, :, k].std():.0f})")
for k, nom in enumerate(["Y", "Cb", "Cr"]):
    axes[1, k].imshow(ycc[:, :, k], cmap="gray", vmin=0, vmax=255)
    axes[1, k].set_title(f"{nom} (écart type {ycc[:, :, k].std():.0f})")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Même image, deux bases : en RVB les trois canaux se ressemblent, en YCbCr presque tout est dans Y",
             color=LIOUAI["navy"]); plt.tight_layout(); plt.show()

**Activité** : En RVB, les trois canaux sont presque la même image : ils sont très **redondants**. Que deviennent les détails (les yeux, les galons de l'uniforme) dans les canaux $C_b$ et $C_r$ ?

Indiquez vos réponses et justifiez-les.

### Les deux méthodes, à place égale

L'image originale occupe trois canaux pleins. Les deux méthodes en gardent l'équivalent d'un canal et demi.

| Méthode | Ce qu'on garde | Place |
|---|---|---|
| Inès | $R$, $G$, $B$ avec une colonne sur deux | $3 \times \frac{1}{2} = 1{,}5$ canal |
| Yanis (JPEG « 4:2:0 ») | $Y$ complet ; $C_b$, $C_r$ avec un pixel sur quatre (blocs $2 \times 2$) | $1 + 2 \times \frac{1}{4} = 1{,}5$ canal |

Pour comparer objectivement, on utilise le **PSNR** (rapport signal sur bruit de crête), en décibels : plus il est grand, plus l'image reconstruite est proche de l'originale. Au-dessus de 40 dB, la différence est en général invisible ; vers 30 dB, elle se remarque.

$$
\text{PSNR} = 10 \log_{10} \frac{255^2}{\text{EQM}}, \qquad \text{EQM} = \text{moyenne des } (\text{pixel reconstruit} - \text{pixel original})^2
$$

In [ ]:
def reduire_colonnes(canal):
    # moyenne de deux colonnes voisines, puis on recopie pour revenir à la taille de départ
    h, w = canal.shape
    petit = canal.reshape(h, w // 2, 2).mean(axis=2)
    return np.repeat(petit, 2, axis=1)

def reduire_blocs(canal):
    # moyenne de chaque bloc 2x2, puis on recopie
    h, w = canal.shape
    petit = canal.reshape(h // 2, 2, w // 2, 2).mean(axis=(1, 3))
    return np.repeat(np.repeat(petit, 2, axis=0), 2, axis=1)

def psnr(image, reference):
    eqm = np.mean((np.clip(image, 0, 255) - reference) ** 2)
    return 10 * np.log10(255 ** 2 / eqm)

ines = np.dstack([reduire_colonnes(photo[:, :, k]) for k in range(3)])
ycc_reduit = np.dstack([ycc[:, :, 0], reduire_blocs(ycc[:, :, 1]), reduire_blocs(ycc[:, :, 2])])
yanis = ycbcr_vers_rvb(ycc_reduit)

print(f"Méthode d'Inès  : PSNR = {psnr(ines, photo):.1f} dB")
print(f"Méthode de Yanis : PSNR = {psnr(yanis, photo):.1f} dB")

zone = (slice(150, 330), slice(150, 360))       # le visage
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, im, titre in zip(axes, [photo, ines, yanis], ["Originale", "Inès (RVB réduit)", "Yanis (YCbCr 4:2:0)"]):
    ax.imshow(np.clip(im[zone], 0, 255).astype(np.uint8)); ax.set_title(titre); ax.axis("off")
plt.tight_layout(); plt.show()

### Les vecteurs de la nouvelle base

Les colonnes de $M^{-1}$ sont les vecteurs de la base YCbCr, exprimés en RVB. Ajoutons chacun d'eux à un gris moyen pour voir « dans quelle direction » il pousse la couleur.

In [ ]:
M_inv = np.linalg.inv(M)
print("Colonnes de M^-1 (vecteurs de base, en coordonnées RVB) :")
for k, nom in enumerate(["Y", "Cb", "Cr"]):
    print(f"  {nom:2s} : {M_inv[:, k].round(3) + 0.0}")

gris = np.array([128., 128, 128])
fig, axes = plt.subplots(1, 3, figsize=(9, 2.4))
for ax, k, nom in zip(axes, range(3), ["Y", "Cb", "Cr"]):
    bande = np.array([[np.clip(gris + t * M_inv[:, k], 0, 255) for t in np.linspace(-60, 60, 13)]])
    ax.imshow(bande.astype(np.uint8), aspect="auto"); ax.set_yticks([])
    ax.set_xticks([0, 6, 12], ["−60", "0", "+60"]); ax.set_title(f"gris + t × (vecteur {nom})")
plt.tight_layout(); plt.show()

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_meilleure_methode}, écart {pari_ecart_qualite}")
print(f"Écart mesuré : {psnr(yanis, photo) - psnr(ines, photo):.1f} dB en faveur de Yanis")

**Activité** :

1. Rédigez la réponse au club en trois phrases : quelle méthode garder, et pourquoi elle marche, sans employer le mot « matrice ».
2. Le vecteur de base de $Y$ est $(1\,;\,1\,;\,1)$ : que signifie le fait qu'il soit gris ?
3. Le changement de base, à lui seul, ne fait gagner **aucune** place : il y a toujours trois nombres par pixel. D'où vient le gain, alors ?
4. Un écart de quelques décibels est-il important ? Comparez avec les trois images zoomées.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Les arrondis.** Une image est stockée en entiers de 0 à 255. Si l'on arrondit $Y$, $C_b$, $C_r$ à l'entier puis qu'on revient en RVB, l'aller-retour n'est plus exact. Quelle erreur maximale cela produit-il ?
- **Une autre base.** Rien n'oblige à choisir YCbCr. Écrivez une fonction `coordonnees(pixels, P)` qui donne les coordonnées de pixels (tableau `(n, 3)`) dans la base formée des colonnes de $P$, **en résolvant un système** plutôt qu'en inversant $P$. Vérifiez-la avec $P = M^{-1}$ : on doit retrouver $Y$, $C_b - 128$, $C_r - 128$.
- **Généraliser.** Testez la méthode de Yanis avec la base « naïve » $(R\,;\,G - R\,;\,B - R)$, qui garde $R$ en pleine résolution : est-ce aussi bon ? Qu'est-ce qui rend la base YCbCr meilleure pour l'œil ?

In [ ]:
def coordonnees(pixels, P):
    # À compléter : renvoie un tableau (n, 3) des coordonnées dans la base des colonnes de P
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def coordonnees(pixels, P):
    # P c = v pour chaque pixel ; on résout tous les systèmes d'un coup (une colonne par pixel)
    return np.linalg.solve(P, pixels.T).T

pixels = photo.reshape(-1, 3)
c = coordonnees(pixels, np.linalg.inv(M))
print("Même résultat que la formule JPEG ?", np.allclose(c, (ycc - decalage).reshape(-1, 3)))

# Arrondis : stockage en entiers
ycc_entiers = np.clip(np.round(ycc), 0, 255)
retour_entiers = np.round(ycbcr_vers_rvb(ycc_entiers))
print("Erreur maximale après arrondi :", np.abs(retour_entiers - photo).max(), "niveau(x) sur 255")

# Base naïve : R, G - R, B - R
P_naive = np.array([[1., 0, 0], [1, 1, 0], [1, 0, 1]])      # colonnes : vecteurs de base en RVB
c = coordonnees(pixels, P_naive).reshape(photo.shape)
c_reduit = np.dstack([c[:, :, 0], reduire_blocs(c[:, :, 1]), reduire_blocs(c[:, :, 2])])
naive = c_reduit @ P_naive.T
print(f"Base naïve (R complet) : PSNR = {psnr(naive, photo):.1f} dB  ; YCbCr : {psnr(yanis, photo):.1f} dB")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Repérer ressemblances et différences entre situations pour abstraire": 1,
    "Comparer les mérites des concepts disponibles": 1,
    "Décrire le concept": 1,
    "Relier les traits du résultat à leur sens réel": 1,
    "Expliquer visuellement": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "5.1-changer-de-base-couleurs", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")